In [2]:
# import time
# time.sleep(1800)

In [3]:
# pip install --upgrade pyarrow shapely geopandas


---

### **4. Estadísticas Descriptivas.ipynb**

  - **4.2. Cálculo de estadísticas:** 
    - **4.2.1.** Estadísticas descriptivas para los datos monetarios ajustados.
    - **4.2.2.** Procesamiento y guardado de diversos resultados estadísticos.


In [4]:
# 1. Módulos y Bibliotecas
# Importar todos los módulos y bibliotecas necesarios.
import pandas as pd
import os
from funciones import * #generate_Qs_from_year, sintetizar_datos


In [5]:
# 2. Configuración de Directorios y Parámetros
# Establecer y verificar las rutas de los directorios.
# Definir los parámetros necesarios.
if not os.path.exists('./../data/Pobreza/'):
    os.makedirs('./../data/Pobreza/')

# -------------------
# Parameters and Configuration
# -------------------

FRAC = 0.02
START_YEAR = 2023
END_YEAR = 2025
EXPERIMENT_TAG = 'ARG'

PATH_POBREZA = './../data/Pobreza/'



In [6]:
# # 3. Exploración y Preprocesamiento de Datos
# # Cargar conjuntos de datos.
# # Explorar datos iniciales.
# # Preparar datos para síntesis.

# # Carga de Datos Auxiliares: Datos del adulto equivalente y Canasta básica regional deflactada
# ad_eq = pd.read_csv('./../data/info/adulto_eq.csv')
# url_canasta_q_deflac = 'https://raw.githubusercontent.com/matuteiglesias/canastasINDEC/main/data/CB_Reg_defl_Q.csv'
# CB_ipc = pd.read_csv(url_canasta_q_deflac)
# print(CB_ipc.tail())

# # Preparación de la Información Geográfica
# radio_ref = pd.read_csv('./../data/info/radio_ref.csv', usecols = ['RADIO_REF_ID', 'DPTO', 'FRAC_REF_ID', 'NOMDPTO', 'radio'])
# dpto_region = pd.read_csv('./../data/info/DPTO_PROV_Region.csv')
# radio_ref = radio_ref.merge(dpto_region)
# radio_ref['COD_2010'] = radio_ref['radio'].astype(str).str.zfill(9)
# radio_ref = radio_ref.drop('radio', axis = 1)
# radio_ref_cols = radio_ref

# claves_dptos = pd.read_csv('https://raw.githubusercontent.com/matuteiglesias/elecciones-ARG/main/datos/BD/claves_dptos_ref.csv')
# claves_dptos_cols = claves_dptos[['distrito_id', 'seccion_id', 'IN1', 'NAM']].drop_duplicates()
# display('dtypes claves_dptos_ref', claves_dptos_cols.dtypes)

# radios_circuitos_secciones_ref = pd.read_csv('./../../CNE-INDEC-georef/info/radios_circuitos_secciones_ref.csv')
# radios_circuitos_secciones_ref = radios_circuitos_secciones_ref[['COD_2010', 'distrito_id', 'seccion_id', 'seccion_nombre', 'circuito']]
# display('radios_circuitos_secciones_ref', radios_circuitos_secciones_ref.max())

# DPTO_Region = radio_ref[['DPTO', 'Region']].drop_duplicates()


## **Estadisticas Descriptivas**

In [7]:


# # Nombres de los archivos
# personas_ingresos_Q_file = f'./../data/Pobreza/personas_ingresos_f{FRAC}_{Q}_{EXPERIMENT_TAG}.csv'
# pobreza_hogares_file = f'./../data/Pobreza/pobreza_hogares_f{FRAC}_q{Q}.csv'
# hogares_geo_file = f'./../data/Pobreza/hogares_geo_f{FRAC}_{Q.split("-")[0]}_{EXPERIMENT_TAG}.csv'

# # Cargar los archivos (solo las primeras 5 filas)
# personas_ingresos_Q = pd.read_csv(personas_ingresos_Q_file, nrows=5)
# pobreza_hogares = pd.read_csv(pobreza_hogares_file, nrows=5)
# hogares_geo = pd.read_csv(hogares_geo_file, nrows=5)

# # Merges
# info_personas = personas_ingresos_Q.merge(pobreza_hogares, on=['HOGAR_REF_ID', 'Q'], how='left').merge(hogares_geo, on='HOGAR_REF_ID', how='left')
# info_hogares = pobreza_hogares.merge(hogares_geo, on='HOGAR_REF_ID', how='left')

# # Mostrar columnas
# print("Columnas de info_personas:", info_personas.columns)
# print("\nColumnas de info_hogares:", info_hogares.columns)


# %% [markdown]
# ## **4. Preparación de Datos para Síntesis**

# %%
# # Lista de agrupaciones para las transformaciones
# groupersP = [['Q', 'Total']]
# groupersH = [['Q', 'Total']]

groupersP = [ # ['Q', 'distrito_id',	'seccion_id', 'seccion_nombre', 'circuito'], 
    ['Q','Total'], ['Q','AGLOSI'], ['Q','AGLOMERADO'], ['Q','Region'], ['Q','PROV'], ['Q','DPTO'], ['Q','P0910'], 
             ['Q','Region', 'AGLOSI'], ['Q', 'PROV', 'AGLOSI']]

groupersH = [['Q','Total'], ['Q','AGLOSI'], ['Q','AGLOMERADO'], ['Q','Region'], ['Q','PROV'], ['Q','DPTO'],
            #  ]#,
             ['Q','Region', 'AGLOSI'], ['Q', 'PROV', 'AGLOSI']]
            

In [8]:
## A pesos actuales
from datetime import datetime

# Cargar el índice de precios al consumidor (CPI) desde la fuente
cpi = pd.read_csv('https://raw.githubusercontent.com/matuteiglesias/IPC-Argentina/main/data/info/indice_precios_M.csv', index_col=0)
cpi.index = pd.to_datetime(cpi.index)

# Obtener la fecha de hoy en formato año-mes
hoy = datetime.today().strftime('%Y-%m')

# Calcular el ratio de precios de hoy con respecto a los precios con índice en base al modelo
ix = cpi.loc[hoy, 'index'].values[0] / cpi.loc['2016-01', 'index'].values[0]

# Lista de columnas relacionadas con montos en pesos
columnas_pesos = ['P47T_persona', 'P47T_hogar', 'CBA', 'gap_indigencia', 'CBT', 'gap_pobreza']

## Print log of the current day and index compared to base, format 2 decimals, unit is base 01-01-2016
print(f'Fecha de hoy: {hoy}')
print(f'Índice de precios al consumidor: {ix:.2f} (base 2016-01 = 1)')



Fecha de hoy: 2024-09
Índice de precios al consumidor: 91.61 (base 2016-01 = 1)


In [9]:
# existing_data

In [12]:
from pd.errors import ParserError  # Import ParserError


In [13]:

# %%
from numpy import power
import os
results_path = './../data/results'

# Create dictionaries for each base universe
all_data = {
    'P': {},
    'PAGLO': {},
    'M24': {},
    'H': {},
    'Hp': {},
    'Hi': {}
}



# Loop through the years
for yr in range(START_YEAR, END_YEAR):
# for yr in range(2015, 2016):
    yr = str(yr)  # Convert year to string for consistency with quarters
    print(f"Processing data for year {yr}...")

    # Procesamiento de datos para cada trimestre del año en cuestión.
    relevant_quarters = generate_Qs_from_year(yr)
    for Q in relevant_quarters:
        # LOADING AND MERGING    
        print(f"Loading data for Q: {Q}")
        # Nombres de los archivos dependientes de Q
        personas_ingresos_Q_file = f'{PATH_POBREZA}individual_income_sample{FRAC}_q{Q}_{EXPERIMENT_TAG}.csv'
        pobreza_hogares_file = f'{PATH_POBREZA}household_poverty_sample{FRAC}_q{Q}_{EXPERIMENT_TAG}.csv'
        hogares_geo_file = f'{PATH_POBREZA}geo_households_sample{FRAC}_{Q.split("-")[0]}_{EXPERIMENT_TAG}.csv'

        # Check if the file exists
        if not os.path.exists(personas_ingresos_Q_file):
            print(f"Warning: {personas_ingresos_Q_file} does not exist. Skipping Q: {Q}")
            continue
        
        # Cargar los archivos
        personas_ingresos_Q = pd.read_csv(personas_ingresos_Q_file)
        pobreza_hogares = pd.read_csv(pobreza_hogares_file)
        hogares_geo = pd.read_csv(hogares_geo_file)

        # Fusiones para obtener los datasets consolidados
        print(f"Merging data for Q: {Q}")
        info_personas = personas_ingresos_Q.merge(pobreza_hogares, on=['HOGAR_REF_ID', 'Q'], how='left'
                                                ).merge(hogares_geo, on='HOGAR_REF_ID', how='left')
        info_hogares = pobreza_hogares.merge(hogares_geo, on='HOGAR_REF_ID', how='left')

        # Agregar AGLO SI, y Total Pais.
        info_personas['AGLOSI'] = info_personas.AGLOMERADO != 0
        info_personas['Total'] = True
        info_hogares['AGLOSI'] = info_hogares.AGLOMERADO != 0
        info_hogares['Total'] = True


        ## Adaptar datos (Pesos actuales, AGLOS si, IDFRAC)
        for df in [info_personas, info_hogares]:
            for col in columnas_pesos:
                if col in df.columns: 
                    if col == 'P47T_persona': 
                        print('Lin to log for ', col, '. Current mean: ', df[col].mean())
                        df[col] = power(10, df[col]) - 1   ## Log -> Lin
                    if col in ['CBA', 'CBT', 'CB_EQUIV']: 
                        print('Check for ', col, '. Mean: ', df[col].mean())
                        print('Check for ', col, '. Median: ', df[col].median())
                    
                    df[col] = (ix*df[col]).round(-1).astype(int)  ## Pesos actuales


        # Define mapping of base_str to its dataset and associated grouper
        mappings = {
            'P': (info_personas, groupersP),
            'PAGLO': (info_personas[info_personas.AGLOSI], groupersP),
            'M24': (info_personas[info_personas.P03 >= 24], groupersP),
            # 'M18': (info_personas[info_personas.P03 >= 18], groupersP),
            'H': (info_hogares, groupersH),
            'Hp': (info_hogares[info_hogares.Pobreza], groupersH),
            'Hi': (info_hogares[info_hogares.Indigencia], groupersH)
        }


        # TRANSFORMACIONES
        print(f"Transforming data for Q: {Q}")
        for base_str, (data_subset, grouper_list) in mappings.items():
            # Create a separate dictionary for the current base string

            for grouper in grouper_list:
                # Synthesize data
                new_data = sintetizar_datos(data_subset, grouper, base_str, FRAC)
                # timestamp = dt.datetime.now().isoformat()
                
                # Construct filename
                filename = f'{results_path}/stats_{base_str}_{"-".join(grouper)}_sample{FRAC}.csv'
                
                # If file exists, read and append data
                if os.path.exists(filename):
                    try:
                        existing_data = pd.read_csv(filename)
                    except ParserError:
                        existing_data = pd.read_csv(filename, on_bad_lines='skip')
                        print('Warning: Parser Error with file ', filename)

                    combined_data = pd.concat([existing_data, new_data], axis=0)
                    df = combined_data
                else:
                    # If file does not exist, simply save the new data
                    df = new_data
                    
                
                # Ensure DataFrame isn't empty before proceeding
                if df.empty:
                    print("Warning: DataFrame is empty after dropping NaNs.")
                else:
                    # Convert 'timestamp' to datetime
                    df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
                    
                    # Drop rows where 'timestamp' could not be converted
                    df = df.dropna(subset=['timestamp'])

                    # Ensure there's data left after dropping NaNs
                    if df.empty:
                        print("Warning: DataFrame is empty after dropping NaNs.")
                    else:
                        # Updated approach to avoid the warning:
                        df = df.copy()  # Ensures df is a copy, not a view
                        df['latest'] = df.groupby(df.columns.difference(['valor', 'timestamp']).tolist())['timestamp'].transform('max')
                        result = df[df['timestamp'] == df['latest']].drop(columns=['latest']).drop_duplicates()

                # Save the result to CSV
                print(filename)
                if os.path.exists(filename):
                    # Append to the existing file without writing the header
                    result.to_csv(filename, mode='a', header=False, index=False)
                else:
                    # Write to a new file with the header
                    result.to_csv(filename, index=False)


                # # Group by all columns except 'valor' and 'timestamp', and get the index of the latest timestamp
                # # df['timestamp'] = pd.to_datetime(df['timestamp'])
                # df = df.dropna(subset=['timestamp'])
                # df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

                # idx = df.groupby(df.columns.difference(['valor', 'timestamp']).tolist())['timestamp'].idxmax()
        
                # # Filter the dataframe using these indices
                # result = df.loc[idx].drop_duplicates()
                # print(filename)

                # result.to_csv(filename, index=False)


# %%
pd.set_option('display.float_format', '{:.2f}'.format)



Processing data for year 2023...
Loading data for Q: 2023-02-15
Merging data for Q: 2023-02-15
Lin to log for  P47T_persona . Current mean:  2.280175472962078
Check for  CBA . Mean:  8751.581426975912
Check for  CBA . Median:  4916.61
Check for  CBT . Mean:  19364.581416034547
Check for  CBT . Median:  10866.95
Check for  CBA . Mean:  3984.2839264418876
Check for  CBA . Median:  3654.17
Check for  CBT . Mean:  8828.319110696584
Check for  CBT . Median:  8076.34
Transforming data for Q: 2023-02-15
./../data/results/stats_P_Q-Total_sample0.02.csv
./../data/results/stats_P_Q-AGLOSI_sample0.02.csv
./../data/results/stats_P_Q-AGLOMERADO_sample0.02.csv
./../data/results/stats_P_Q-Region_sample0.02.csv
./../data/results/stats_P_Q-PROV_sample0.02.csv
./../data/results/stats_P_Q-DPTO_sample0.02.csv
./../data/results/stats_P_Q-P0910_sample0.02.csv
./../data/results/stats_P_Q-Region-AGLOSI_sample0.02.csv
./../data/results/stats_P_Q-PROV-AGLOSI_sample0.02.csv
./../data/results/stats_PAGLO_Q-Total_

In [ ]:
# pip install --upgrade pandas numpy

In [ ]:
# x = info_personas.head().copy()
# x.loc[:, 'A'] = 1





In [ ]:
# # info_personas['AGLOSI'] = 1


# info_personas = info_personas.head()
# info_personas['AGLOSI'] = 1


In [ ]:
# df = df.dropna(subset=['timestamp'])
# df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

# cols = df.columns.difference(['valor', 'timestamp']).tolist()
# print(cols)
# df.groupby(cols)['timestamp'].idxmax()

# # ['Q', 'Total', 'base', 'frac', 'observable', 'sintetico']


# # 0    2024-08-26 18:57:50.796257
# # 1    2024-08-26 18:57:50.796257
# # 2    2024-08-26 18:57:50.796257
# # 3    2024-08-26 18:57:50.796257
# # 4    2024-08-26 18:57:50.796257
# # 5    2024-08-26 18:57:50.796257
# # 6    2024-08-26 18:57:50.796257
# # 7    2024-08-26 18:57:50.796257
# # 8    2024-08-26 18:57:50.796257
# # 9    2024-08-26 18:57:50.796257
# # 10   2024-08-26 18:57:50.796257
# # 11   2024-08-26 18:57:50.796257